In [1]:
 """SESSION 21 — Mask2Former with Swin-L backbone and mask-aware TTA.
"""

from __future__ import annotations

# %% [markdown]
# # SESSION 21 — Mask2Former Swin-L + scale/flip TTA

# %% 1. Imports and dependency bootstrap
import argparse
import hashlib
import json
import math
import multiprocessing
import os
import random
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlretrieve

# Set before any torch import. The subprocess-based dependency probe below keeps
# torch out of this notebook process until installation is complete.
# PyTorch 2.4+ uses PYTORCH_CUDA_ALLOC_CONF; older versions used PYTORCH_ALLOC_CONF.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_ALLOC_CONF",      "expandable_segments:True")

SESSION_NAME = "session21_mask2former_swin_l_tta"
RUNTIME_BUILD = "SESSION21-MASK2FORMER-SWIN-L-V7"
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

MMCV_INDEX = "https://download.openmmlab.com/mmcv/dist/cu121/torch2.4.0/index.html"
TORCH_INDEX = "https://download.pytorch.org/whl/cu121"
COCO_SWIN_L_URL = (
    "https://download.openmmlab.com/mmdetection/v3.0/mask2former/"
    "mask2former_swin-l-p4-w12-384-in21k_16xb1-lsj-100e_coco-panoptic/"
    "mask2former_swin-l-p4-w12-384-in21k_16xb1-lsj-100e_coco-panoptic_"
    "20220407_104949-82f8d28d.pth"
)


def _pip(*args: str) -> None:
    print(f"  [install] {' '.join(args)}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *args]
    )


def _ensure_supported_runtime() -> bool:
    """Re-launch on Python 3.11 when Kaggle's Python 3.13 has no MMCV wheel.

    OpenMMLab does not provide a CPython-3.13 wheel for MMCV 2.2.0. A local
    Python 3.11 environment uses the published cu121/torch2.4 MMCV wheel.
    Returns True only in the parent notebook process after its child completed.
    """
    if os.environ.get("SESSION21_PY311_RUNTIME") == "1":
        return False
    if sys.version_info[:2] < (3, 13):
        return False

    source_value = globals().get("__file__")
    source = Path(source_value).resolve() if source_value else None
    if source is None or not source.is_file():
        # A pasted Kaggle/Papermill cell has no __file__. Reconstruct the
        # complete current cell so the Python-3.11 child receives a real file.
        runtime = Path("/kaggle/working/SESSION_21_runtime.py")
        recovered = ""
        try:
            import textwrap
            from IPython import get_ipython
            shell = get_ipython()
            history = getattr(getattr(shell, "history_manager", None),
                              "input_hist_raw", [])
            recovered = next(
                (textwrap.dedent(cell).lstrip() for cell in reversed(history)
                 if f'RUNTIME_BUILD = "{RUNTIME_BUILD}"' in cell
                 and "def main" in cell
                 and "_ensure_supported_runtime" in cell
                 and "if __name__ == \"__main__\":" in cell),
                "",
            )
        except Exception as error:
            history_error = str(error)
        else:
            history_error = ""
        if not recovered and runtime.is_file():
            candidate = runtime.read_text(encoding="utf-8", errors="replace")
            if f'RUNTIME_BUILD = "{RUNTIME_BUILD}"' in candidate:
                recovered = candidate
        if not recovered:
            raise RuntimeError(
                "Session 21 could not recover its complete notebook source "
                f"({history_error or 'no current source cell found'}). "
                "Use the regenerated SESSION_21_mask2former_swin_l_tta.ipynb "
                "or paste the complete updated Session 21 source." 
            )
        try:
            compile(recovered, str(runtime), "exec")
        except SyntaxError as error:
            raise RuntimeError("Recovered Session 21 notebook source is invalid Python.") from error
        runtime.parent.mkdir(parents=True, exist_ok=True)
        temporary = runtime.with_suffix(".py.tmp")
        temporary.write_text(recovered, encoding="utf-8")
        temporary.replace(runtime)
        print(f"  [runtime] wrote notebook source to {runtime}", flush=True)
        source = runtime
    runtime_root = Path("/kaggle/working/session21_openmmlab_py311")
    if not runtime_root.parent.is_dir():
        raise RuntimeError("Python 3.13 is unsupported for Session 21; run it on Kaggle.")
    python = runtime_root / "bin" / "python"
    has_pip = python.is_file() and subprocess.run(
        [str(python), "-m", "pip", "--version"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    ).returncode == 0
    if not has_pip:
        # ``uv venv`` deliberately omits pip unless --seed is requested. Clear
        # the incomplete V5 environment if present, then seed pip/setuptools/
        # wheel so all later package installs work through this interpreter.
        print("  [runtime] creating seeded isolated Python 3.11 environment ...", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-input", "uv"])
        subprocess.check_call([
            sys.executable, "-m", "uv", "venv", "--clear", "--seed",
            "--python", "3.11", str(runtime_root),
        ])
    if subprocess.run([str(python), "-m", "pip", "--version"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
        raise RuntimeError("Session 21 could not create a Python 3.11 environment with pip.")

    environment = os.environ.copy()
    environment["SESSION21_PY311_RUNTIME"] = "1"
    environment["PYTHONNOUSERSITE"] = "1"
    environment["MPLBACKEND"] = "Agg"
    command = [str(python), "-u", str(source), *sys.argv[1:]]
    print("  [runtime] re-launching Session 21 with Python 3.11 ...", flush=True)
    completed = subprocess.run(
        command, env=environment, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    if completed.stdout:
        print(completed.stdout, end="", flush=True)
    if completed.returncode:
        raise RuntimeError(
            "Session 21 Python-3.11 runtime failed with exit status "
            f"{completed.returncode}. The child traceback is printed above."
        )
    return True


def _patch_mmdet_mmcv_guard() -> None:
    """Permit MMCV 2.2.0 in MMDetection 3.3 for this tested runtime."""
    import importlib.util

    spec = importlib.util.find_spec("mmdet")
    if spec is None or spec.origin is None:
        raise RuntimeError("mmdet was installed but its package path is missing")
    init_path = Path(spec.origin)
    source = init_path.read_text()
    old = "mmcv_maximum_version = '2.2.0'"
    if old in source:
        init_path.write_text(source.replace(
            old, "mmcv_maximum_version = '2.3.0'", 1
        ))
        print(f"  [deps] adjusted MMDetection's MMCV guard: {init_path}")


def _dependency_probe() -> str:
    # torchvision removed from probe — not required by MMDet and may be
    # version-mismatched after torch downgrade to 2.5.1 on Python 3.13
    command = (
        "import torch, mmcv, mmengine, mmdet; "
        "from mmcv.ops import MultiScaleDeformableAttention, batched_nms; "
        "from mmdet.models import Mask2Former; "
        "print(torch.__version__, mmcv.__version__, "
        "mmengine.__version__, mmdet.__version__)"
    )
    return subprocess.check_output(
        [sys.executable, "-c", command],
        text=True,
        stderr=subprocess.STDOUT,
        env={**os.environ, "MPLBACKEND": "Agg"},
    ).strip()


def _ensure_dependencies() -> None:
    """Install the Python-3.11 / Torch-2.4 / CUDA-12.1 OpenMMLab stack."""
    try:
        versions = _dependency_probe()
        print(f"  [deps] usable environment found: {versions}")
    except (subprocess.CalledProcessError, OSError) as err:
        detail = getattr(err, "output", "")
        if detail:
            print("  [deps] current environment probe failed:")
            print("\n".join(detail.strip().splitlines()[-6:]))
        if sys.version_info[:2] != (3, 11):
            raise RuntimeError(
                "Session 21 requires the isolated Python 3.11 runtime. "
                "Run the regenerated Kaggle notebook from its first cell."
            )
        print("  [deps] installing isolated Python-3.11 OpenMMLab packages ...")
        # mmengine 0.10.5 still imports pkg_resources; setuptools 81+ removed it.
        _pip("--upgrade", "pip", "setuptools==75.1.0", "wheel")
        _pip("--upgrade", "--force-reinstall", "--no-cache-dir",
             "torch==2.4.0", "torchvision==0.19.0", "--index-url", TORCH_INDEX)
        _pip("--upgrade", "numpy<2", "addict", "termcolor", "packaging",
             "opencv-python-headless==4.10.0.84", "matplotlib", "pyyaml",
             "Pillow", "pycocotools", "shapely", "six", "terminaltables", "tqdm",
             "scipy", "rich", "yapf")
        _pip("--upgrade", "--no-deps", "mmengine==0.10.5")
        _pip(
            "--upgrade", "--no-cache-dir", "--only-binary=mmcv", "--no-deps",
            "mmcv==2.2.0", "-f", MMCV_INDEX,
        )
        _pip("--upgrade", "--no-deps", "mmdet==3.3.0")
        _patch_mmdet_mmcv_guard()

        try:
            versions = _dependency_probe()
        except subprocess.CalledProcessError as probe_error:
            print("  [deps] post-install probe failed:")
            print((probe_error.output or "").strip())
            raise
        print(f"  [deps] verified in a fresh interpreter: {versions}")


# %% 2. Dataset discovery and YOLO-polygon to COCO conversion
def _safe_extract_dataset_zip(archive: Path) -> Path:
    """Extract an attached YOLO dataset ZIP to working storage exactly once."""
    archive = archive.resolve()
    if not archive.is_file() or archive.suffix.lower() != ".zip":
        raise FileNotFoundError(f"Dataset ZIP does not exist: {archive}")
    cache_base = (Path("/kaggle/working") if Path("/kaggle/working").is_dir()
                  else archive.parent) / "session21_dataset_cache"
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()[:16]
    destination = cache_base / digest
    marker = destination / ".complete"
    if not marker.is_file():
        with zipfile.ZipFile(archive) as zipped:
            broken = zipped.testzip()
            if broken:
                raise ValueError(f"Corrupt dataset ZIP member: {broken}")
            destination.mkdir(parents=True, exist_ok=True)
            root = destination.resolve()
            for member in zipped.infolist():
                member_path = Path(member.filename)
                target = (destination / member_path).resolve()
                if (member_path.is_absolute() or ".." in member_path.parts
                        or not target.is_relative_to(root)):
                    raise ValueError(f"Unsafe ZIP member: {member.filename}")
                if (member.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError(f"Dataset ZIP contains symlink: {member.filename}")
            zipped.extractall(destination)
        marker.write_text("ok\n")
    candidates = [p.parent for p in destination.rglob("data.yaml")
                  if _valid_dataset_root(p.parent)]
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one complete YOLO dataset in {archive.name}; found {len(candidates)}"
        )
    resolved = candidates[0].resolve()
    print(f"  [data] extracted {archive.name} to: {resolved}")
    return resolved


def _find_dataset_root(hint: str) -> Path:
    if hint != "auto":
        explicit = Path(hint).expanduser().resolve()
        if explicit.suffix.lower() == ".zip":
            return _safe_extract_dataset_zip(explicit)
        if _valid_dataset_root(explicit):
            print(f"  [data] explicit rebuilt dataset: {explicit}")
            return explicit
        raise FileNotFoundError(
            f"--data-root is not a complete rebuilt dataset or ZIP: {explicit}"
        )

    # Explicit Kaggle personal-dataset path (highest priority)
    _EXPLICIT = Path("/kaggle/input/datasets/munnamuntasirul/skin-disease-yolo/skin_disease_yolo")
    if _EXPLICIT.exists() and _valid_dataset_root(_EXPLICIT):
        print(f"  [data] found at explicit path: {_EXPLICIT}")
        return _EXPLICIT.resolve()

    preferred = [
        Path("/kaggle/working/skin_disease_yolo"),
        Path("skin_disease_yolo"),
        Path("../skin_disease_yolo"),
    ]
    for candidate in preferred:
        if _valid_dataset_root(candidate):
            resolved = candidate.resolve()
            print(f"  [data] found rebuilt dataset: {resolved}")
            return resolved

    kaggle_input = Path("/kaggle/input")
    zip_roots = [Path("/kaggle/working"), kaggle_input]
    for zip_root in zip_roots:
        if zip_root.is_dir():
            archives = sorted(zip_root.rglob("skin_disease_yolo.zip"))
            if len(archives) == 1:
                return _safe_extract_dataset_zip(archives[0])
            if len(archives) > 1:
                raise RuntimeError(
                    "Multiple skin_disease_yolo.zip files found. Pass --data-root explicitly."
                )
    if kaggle_input.is_dir():
        named = sorted(kaggle_input.rglob("skin_disease_yolo"))
        for candidate in named:
            if _valid_dataset_root(candidate):
                resolved = candidate.resolve()
                print(f"  [data] found named Kaggle dataset: {resolved}")
                return resolved

        layout_matches = []
        for data_yaml in sorted(kaggle_input.rglob("data.yaml")):
            if _valid_dataset_root(data_yaml.parent):
                layout_matches.append(data_yaml.parent.resolve())
        if len(layout_matches) == 1:
            print(f"  [data] found Kaggle dataset layout: {layout_matches[0]}")
            return layout_matches[0]
        if len(layout_matches) > 1:
            choices = "\n".join(f"    - {p}" for p in layout_matches)
            raise RuntimeError(
                "Multiple complete datasets were found. Pass --data-root explicitly:\n"
                + choices
            )

    raise FileNotFoundError(
        "skin_disease_yolo was not found. Attach the rebuilt dataset "
        "to Kaggle or pass --data-root /full/path/skin_disease_yolo"
    )


def _valid_dataset_root(root: Path) -> bool:
    return (
        (root / "data.yaml").is_file()
        and all((root / split / "images").is_dir()
        for split in ("train", "valid"))
        and all((root / split / "labels").is_dir()
        for split in ("train", "valid"))
    )


def _load_class_names(data_yaml: Path) -> list[str]:
    import yaml

    data = yaml.safe_load(data_yaml.read_text()) or {}
    names = data.get("names")
    if isinstance(names, dict):
        names = [names[key] for key in sorted(names, key=lambda x: int(x))]
    if not isinstance(names, list) or not names:
        raise ValueError("data.yaml must contain a non-empty 'names' list or mapping")
    names = [str(name) for name in names]
    # Class name validation removed — works with any dataset (was locked to 4 classes)
    print(f"  [data] loaded {len(names)} classes from data.yaml: {names}")
    return names


def _gather_split(root: Path, split: str) -> list[dict]:
    images_dir = root / split / "images"
    labels_dir = root / split / "labels"
    items = []
    for image in sorted(images_dir.iterdir()):
        if image.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        label = labels_dir / f"{image.stem}.txt"
        if not label.is_file():
            raise FileNotFoundError(f"Missing label for {image}: {label}")
        items.append({"image": image.resolve(), "label": label.resolve()})
    if not items:
        raise ValueError(f"No supported images found in {images_dir}")
    return items


def _item_classes(item: dict) -> set[int]:
    classes = set()
    for line in item["label"].read_text().splitlines():
        values = line.split()
        if values:
            classes.add(int(values[0]))
    return classes


def _balanced_subset(items: list[dict], limit: int, seed: int) -> list[dict]:
    """Small deterministic subset that tries to retain every class."""
    if len(items) <= limit:
        return list(items)
    rng = random.Random(seed)
    shuffled = list(items)
    rng.shuffle(shuffled)
    chosen = []
    used = set()
    for class_id in range(max((max(_item_classes(item), default=-1)
                               for item in items), default=-1) + 1):
        for item in shuffled:
            key = str(item["image"])
            if key not in used and class_id in _item_classes(item):
                chosen.append(item)
                used.add(key)
                break
    for item in shuffled:
        if len(chosen) >= limit:
            break
        key = str(item["image"])
        if key not in used:
            chosen.append(item)
            used.add(key)
    return chosen


def _polygon_area(xs: list[float], ys: list[float]) -> float:
    return 0.5 * abs(sum(
        xs[i] * ys[(i + 1) % len(xs)]
        - xs[(i + 1) % len(xs)] * ys[i]
        for i in range(len(xs))
    ))


def _yolo_to_coco(items: list[dict], names: list[str], destination: Path) -> dict:
    from PIL import Image
    from pycocotools import mask as mask_utils

    images = []
    annotations = []
    annotation_id = 1
    skipped_polygons = 0
    class_instances: Counter = Counter()

    for image_id, item in enumerate(items, start=1):
        with Image.open(item["image"]) as image_handle:
            width, height = image_handle.size
        images.append({
            "id": image_id,
            "file_name": str(item["image"]),
            "width": width,
            "height": height,
        })

        for line_number, line in enumerate(
                item["label"].read_text().splitlines(), start=1):
            values = line.strip().split()
            if not values:
                continue
            if len(values) < 7 or (len(values) - 1) % 2:
                skipped_polygons += 1
                continue
            try:
                class_id = int(values[0])
                coords = [float(value) for value in values[1:]]
            except ValueError:
                skipped_polygons += 1
                continue
            if not all(math.isfinite(value) for value in coords):
                skipped_polygons += 1
                continue
            if not 0 <= class_id < len(names):
                raise ValueError(
                    f"Invalid class {class_id} in {item['label']}:{line_number}"
                )

            xs = [max(0.0, min(1.0, coords[i])) * width
                  for i in range(0, len(coords), 2)]
            ys = [max(0.0, min(1.0, coords[i])) * height
                  for i in range(1, len(coords), 2)]
            area = _polygon_area(xs, ys)
            x0, x1 = min(xs), max(xs)
            y0, y1 = min(ys), max(ys)
            if not math.isfinite(area) or area <= 1.0 or x1 <= x0 or y1 <= y0:
                skipped_polygons += 1
                continue

            segmentation = [coordinate
                            for pair in zip(xs, ys)
                            for coordinate in pair]
            rasterized = mask_utils.frPyObjects([segmentation], height, width)
            if float(mask_utils.area(rasterized).sum()) <= 0.0:
                skipped_polygons += 1
                continue
            annotations.append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": class_id + 1,
                "segmentation": [segmentation],
                "area": float(area),
                "bbox": [float(x0), float(y0), float(x1 - x0), float(y1 - y0)],
                "iscrowd": 0,
            })
            class_instances[class_id] += 1
            annotation_id += 1

    document = {
        "info": {"description": "Session 21 YOLO-to-COCO conversion"},
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": [
            {"id": i + 1, "name": name, "supercategory": "skin"}
            for i, name in enumerate(names)
        ],
    }
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(document))
    print(
        f"  [coco] {destination.name}: {len(images)} images, "
        f"{len(annotations)} instances, {skipped_polygons} invalid polygons skipped"
    )
    return {
        "images": len(images),
        "annotations": len(annotations),
        "skipped_polygons": skipped_polygons,
        "instances_per_class": {
            names[index]: int(class_instances[index]) for index in range(len(names))
        },
    }


# %% 3. MMDetection config and launch helpers
def _mmdet_tool(name: str) -> str:
    import mmdet

    root = Path(mmdet.__file__).resolve().parent
    candidates = [
        root / ".mim" / "tools" / f"{name}.py",
        root / "tools" / f"{name}.py",
        root.parent / "tools" / f"{name}.py",
    ]
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    raise FileNotFoundError(
        f"MMDetection's bundled tools/{name}.py was not found under {root}"
    )


def _detect_gpu_ids(device: str) -> list[str]:
    if device != "auto":
        ids = [part.strip() for part in device.split(",") if part.strip()]
        if device == "cpu" or not ids:
            raise RuntimeError("Mask2Former Swin-L training requires a CUDA GPU")
        return ids

    count = int(subprocess.check_output(
        [sys.executable, "-c", "import torch; print(torch.cuda.device_count())"],
        text=True,
    ).strip())
    if count < 1:
        raise RuntimeError(
            "No CUDA GPU detected. Enable a GPU accelerator in Kaggle settings."
        )
    return [str(index) for index in range(min(2, count))]


def _download_and_prepare_coco_checkpoint(work: Path) -> Path:
    """Download COCO Swin-L weights and remove only the COCO classifier.

    The official checkpoint has 133 COCO panoptic classes. All compatible
    backbone, pixel-decoder, transformer-decoder, query and mask weights are
    retained; only the class projection is reinitialized for four skin classes.
    """
    prepared = work / "pretrained" / "mask2former_swin_l_coco_backbone_decoder.pth"
    if prepared.is_file() and prepared.stat().st_size > 1_000_000:
        print(f"  [weights] prepared checkpoint exists: {prepared}")
        return prepared

    raw = work / "pretrained" / "mask2former_swin_l_coco_raw.pth"
    raw.parent.mkdir(parents=True, exist_ok=True)
    print("  [weights] downloading official Mask2Former Swin-L COCO checkpoint ...")
    try:
        urlretrieve(COCO_SWIN_L_URL, raw)
    except Exception as err:
        raw.unlink(missing_ok=True)
        raise RuntimeError(
            "COCO checkpoint download failed. Enable Kaggle Internet and rerun."
        ) from err
    if raw.stat().st_size <= 1_000_000:
        raw.unlink(missing_ok=True)
        raise RuntimeError("Downloaded COCO checkpoint is unexpectedly small")

    import torch

    checkpoint = torch.load(raw, map_location="cpu", weights_only=False)
    state_dict = checkpoint.get("state_dict", checkpoint)
    classifier_keys = [
        key for key in list(state_dict)
        if "panoptic_head.cls_embed" in key
    ]
    for key in classifier_keys:
        del state_dict[key]
    if not classifier_keys:
        print("  [weights] warning: no COCO classifier keys were found to remove")
    torch.save(checkpoint, prepared)
    raw.unlink(missing_ok=True)
    print(
        f"  [weights] retained COCO weights and removed "
        f"{len(classifier_keys)} incompatible classifier tensors"
    )
    return prepared


def _write_training_config(
    destination: Path,
    train_json: Path,
    valid_json: Path,
    work_dir: Path,
    checkpoint: Path,
    names: list[str],
    gpu_count: int,
    max_iters: int,
    image_size: int,
    accumulation: int,
    seed: int,
) -> None:
    classes = repr(tuple(names))
    palette = repr([(220, 20, 60), (119, 11, 32), (0, 0, 142), (0, 0, 230)])
    validation_interval = max(20 if max_iters <= 100 else 500, max_iters // 10)
    effective_batch = gpu_count * accumulation  # one image/GPU/iteration
    # Swin-L + Mask2Former is numerically sensitive on this noisy 19-class
    # dataset; use a conservative fine-tuning rate for the COCO initialization.
    learning_rate = 5e-5 * effective_batch / 16.0
    milestone_1 = max(1, int(max_iters * 0.85))
    milestone_2 = max(milestone_1 + 1, int(max_iters * 0.95))

    config = f'''# Auto-generated by SESSION 21. Do not edit this generated copy.
_base_ = ['mmdet::mask2former/mask2former_swin-t-p4-w7-224_8xb2-lsj-50e_coco.py']

default_scope = 'mmdet'
metainfo = dict(classes={classes}, palette={palette})
num_classes = {len(names)}
image_size = ({image_size}, {image_size})

batch_augments = [dict(
    type='BatchFixedSizePad', size=image_size,
    img_pad_value=0, pad_mask=True, mask_pad_value=0, pad_seg=False,
)]
data_preprocessor = dict(
    type='DetDataPreprocessor',
    mean=[123.675, 116.28, 103.53], std=[58.395, 57.12, 57.375],
    bgr_to_rgb=True, pad_size_divisor=32,
    pad_mask=True, mask_pad_value=0, pad_seg=False,
    batch_augments=batch_augments,
)

depths = [2, 2, 18, 2]
model = dict(
    data_preprocessor=data_preprocessor,
    backbone=dict(
        _delete_=True,
        type='SwinTransformer', pretrain_img_size=384,
        embed_dims=192, depths=depths, num_heads=[6, 12, 24, 48],
        window_size=12, mlp_ratio=4, qkv_bias=True, qk_scale=None,
        drop_rate=0.0, attn_drop_rate=0.0, drop_path_rate=0.3,
        patch_norm=True, out_indices=(0, 1, 2, 3),
        with_cp=True, convert_weights=True, frozen_stages=-1,
        init_cfg=None,
    ),
    panoptic_head=dict(
        in_channels=[192, 384, 768, 1536],
        num_queries=200,
        num_things_classes=num_classes,
        num_stuff_classes=0,
        loss_cls=dict(class_weight=[1.0] * num_classes + [0.1]),
    ),
    panoptic_fusion_head=dict(
        num_things_classes=num_classes, num_stuff_classes=0,
    ),
    test_cfg=dict(
        panoptic_on=False, semantic_on=False, instance_on=True,
        max_per_image=100, iou_thr=0.8, filter_low_score=True,
    ),
)

# Proven working training pipeline (produced mAP50=0.6409 in the first run).
# CopyPaste + MultiImageMixDataset was tried and consistently broke training
# due to incompatibility with the Mask2Former panoptic base config pipeline.
# Keeping the simple, reliable CocoDataset + standard augmentation pipeline.
train_pipeline = [
    dict(type='LoadImageFromFile', to_float32=True),
    dict(type='LoadAnnotations', with_bbox=True, with_mask=True, poly2mask=True),
    dict(type='RandomFlip', prob=0.5),
    # Avoid RandomCrop with allow_negative_crop: empty masks produce NaN mask
    # losses and make Hungarian assignment's cost matrix infeasible.
    dict(type='Resize', scale=image_size, keep_ratio=True),
    dict(type='Pad', size=image_size, pad_val=dict(img=(0, 0, 0))),
    dict(type='FilterAnnotations', min_gt_bbox_wh=(2.0, 2.0), by_mask=True),
    dict(type='PackDetInputs'),
]
test_pipeline = [
    dict(type='LoadImageFromFile', to_float32=True),
    dict(type='Resize', scale=image_size, keep_ratio=True),
    dict(type='PackDetInputs', meta_keys=(
        'img_id', 'img_path', 'ori_shape', 'img_shape', 'scale_factor')),
]

# _delete_=True: prevents the Mask2Former-Swin-T base config's lsj-specific
# batch_sampler keys from leaking into the merged config alongside our
# InfiniteSampler -> avoids 'sampler is mutually exclusive with batch_sampler'.
train_dataloader = dict(
    _delete_=True,
    batch_size=1, num_workers=2, persistent_workers=True,
    sampler=dict(type='InfiniteSampler', shuffle=True),
    dataset=dict(
        type='CocoDataset', data_root='',
        ann_file={str(train_json)!r}, data_prefix=dict(img=''),
        metainfo=metainfo, filter_cfg=dict(filter_empty_gt=True, min_size=1),
        pipeline=train_pipeline,
    ),
)
val_dataloader = dict(
    _delete_=True,
    batch_size=1, num_workers=2, persistent_workers=True, drop_last=False,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type='CocoDataset', data_root='',
        ann_file={str(valid_json)!r}, data_prefix=dict(img=''),
        metainfo=metainfo, test_mode=True, pipeline=test_pipeline,
    ),
)
test_dataloader = val_dataloader
val_evaluator = dict(
    _delete_=True, type='CocoMetric', ann_file={str(valid_json)!r},
    metric=['bbox', 'segm'], classwise=True,
)
test_evaluator = val_evaluator

backbone_norm = dict(lr_mult=0.1, decay_mult=0.0)
backbone_embed = dict(lr_mult=0.1, decay_mult=0.0)
decoder_embed = dict(lr_mult=1.0, decay_mult=0.0)
custom_keys = {{
    'backbone': dict(lr_mult=0.1, decay_mult=1.0),
    'backbone.patch_embed.norm': backbone_norm,
    'backbone.norm': backbone_norm,
    'absolute_pos_embed': backbone_embed,
    'relative_position_bias_table': backbone_embed,
    'query_embed': decoder_embed,
    'query_feat': decoder_embed,
    'level_embed': decoder_embed,
}}
custom_keys.update({{
    f'backbone.stages.{{stage_id}}.blocks.{{block_id}}.norm': backbone_norm
    for stage_id, blocks in enumerate(depths)
    for block_id in range(blocks)
}})
custom_keys.update({{
    f'backbone.stages.{{stage_id}}.downsample.norm': backbone_norm
    for stage_id in range(len(depths) - 1)
}})

optim_wrapper = dict(
    # AMP can overflow Mask2Former mask logits after several hundred updates
    # on tiny/noisy lesion masks. FP32 is stable here and fits the observed
    # roughly 7 GB/GPU footprint at batch size 1.
    _delete_=True, type='OptimWrapper',
    accumulative_counts={accumulation},
    optimizer=dict(type='AdamW', lr={learning_rate}, weight_decay=0.05,
                   eps=1e-8, betas=(0.9, 0.999)),
    paramwise_cfg=dict(custom_keys=custom_keys, norm_decay_mult=0.0),
    clip_grad=dict(max_norm=0.005, norm_type=2),
)
param_scheduler = [dict(
    type='MultiStepLR', begin=0, end={max_iters}, by_epoch=False,
    milestones=[{milestone_1}, {milestone_2}], gamma=0.1,
)]
train_cfg = dict(
    _delete_=True, type='IterBasedTrainLoop', max_iters={max_iters},
    val_interval={validation_interval},
)
# _delete_=True on val_cfg and test_cfg: the Mask2Former base config
# uses EpochBasedValLoop with extra keys. Without _delete_, those keys
# merge into our ValLoop/TestLoop -> TypeError at runner init.
val_cfg = dict(_delete_=True, type='ValLoop')
test_cfg = dict(_delete_=True, type='TestLoop')

default_hooks = dict(
    logger=dict(type='LoggerHook', interval=50),
    checkpoint=dict(
        type='CheckpointHook', by_epoch=False, interval={validation_interval},
        save_last=True, save_best='coco/segm_mAP', rule='greater',
        max_keep_ckpts=2,
    ),
)
log_processor = dict(type='LogProcessor', window_size=50, by_epoch=False)
randomness = dict(seed={seed}, deterministic=False, diff_rank_seed=True)
env_cfg = dict(
    cudnn_benchmark=True,
    mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0),
    dist_cfg=dict(backend='nccl'),
)
# EMA smooths weights across the last ~5000 iterations, giving a more
# generalising checkpoint than the single best-val snapshot alone.
custom_hooks = [dict(
    type='EMAHook', ema_type='ExpMomentumEMA',
    momentum=0.0002, update_buffers=True, priority=49,
)]
# launcher is overridden to 'pytorch' by torchrun when DDP is used;
# 'none' would bypass torchrun's rank injection and break DDP.
launcher = 'none'
work_dir = {str(work_dir)!r}
load_from = {str(checkpoint)!r}
resume = False
'''
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(config)


def _best_checkpoint(work_dir: Path) -> Path | None:
    best = sorted(
        work_dir.glob("best_coco_segm_mAP_iter_*.pth"),
        key=lambda path: path.stat().st_mtime,
    )
    if best:
        return best[-1]
    last = sorted(
        work_dir.glob("iter_*.pth"), key=lambda path: path.stat().st_mtime
    )
    return last[-1] if last else None


def _run_training(
    config: Path, work_dir: Path, gpu_ids: list[str], resume: bool
) -> None:
    train_tool = _mmdet_tool("train")
    environment = os.environ.copy()
    environment["CUDA_VISIBLE_DEVICES"] = ",".join(gpu_ids)
    port = str(random.randint(20000, 29999))
    if len(gpu_ids) > 1:
        command = [
            sys.executable,
            "-m",
            "torch.distributed.run",
            "--nproc_per_node",
            str(len(gpu_ids)),
            "--master_port",
            port,
            train_tool,
            str(config),
            "--work-dir",
            str(work_dir),
            "--launcher",
            "pytorch",
        ]
    else:
        command = [
            sys.executable,
            train_tool,
            str(config),
            "--work-dir",
            str(work_dir),
        ]
    if resume:
        command.append("--resume")
    print(f"  [train] GPUs={gpu_ids}, command={' '.join(command)}")
    try:
        subprocess.check_call(command, env=environment)
    except subprocess.CalledProcessError as exc:
        # Print the generated config so the actual MMDetection error is visible
        # in Kaggle output even when the subprocess output scrolled away.
        print(f"\n  [error] Training subprocess failed (exit code {exc.returncode})")
        print(f"  [error] Config file: {config}")
        print("  [error] --- GENERATED CONFIG BEGIN ---")
        try:
            print(config.read_text())
        except Exception:
            print("  [error] (could not read config file)")
        print("  [error] --- GENERATED CONFIG END ---")
        raise


# %% 4. Separate-pass mask-aware TTA
def _write_test_config(
    train_config: Path,
    destination: Path,
    test_json: Path,
    prediction_prefix: Path,
    scale: int,
    flip: bool,
) -> None:
    override = f'''

# Test override generated by SESSION 21
test_pipeline = [
    dict(type='LoadImageFromFile', to_float32=True),
    dict(type='Resize', scale=({scale}, {scale}), keep_ratio=True),
    dict(type='RandomFlip', prob={1.0 if flip else 0.0}, direction='horizontal'),
    dict(type='PackDetInputs', meta_keys=(
        'img_id', 'img_path', 'ori_shape', 'img_shape', 'scale_factor',
        'flip', 'flip_direction')),
]
test_dataloader = dict(
    batch_size=1, num_workers=2, persistent_workers=True, drop_last=False,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        _delete_=True, type='CocoDataset', data_root='',
        ann_file={str(test_json)!r}, data_prefix=dict(img=''),
        metainfo=metainfo, test_mode=True, pipeline=test_pipeline,
    ),
)
test_evaluator = dict(
    _delete_=True, type='CocoMetric', ann_file={str(test_json)!r},
    metric=['bbox', 'segm'], format_only=True,
    outfile_prefix={str(prediction_prefix)!r},
)
load_from = None
resume = False
'''
    destination.write_text(train_config.read_text() + override)


def _run_prediction_pass(
    config: Path,
    checkpoint: Path,
    work_dir: Path,
    gpu_id: str,
) -> None:
    test_tool = _mmdet_tool("test")
    environment = os.environ.copy()
    environment["CUDA_VISIBLE_DEVICES"] = gpu_id
    command = [
        sys.executable,
        test_tool,
        str(config),
        str(checkpoint),
        "--work-dir",
        str(work_dir),
    ]
    subprocess.check_call(command, env=environment)


def _flip_prediction_to_original(prediction: dict, width: int) -> dict:
    """Undo horizontal flip in both COCO bbox and segmentation."""
    import numpy as np
    from pycocotools import mask as mask_utils

    output = dict(prediction)
    x, y, w, h = [float(value) for value in output["bbox"]]
    output["bbox"] = [float(width - x - w), y, w, h]
    segmentation = output.get("segmentation")
    if isinstance(segmentation, dict):
        decoded = mask_utils.decode(segmentation)
        if decoded.ndim == 3:
            decoded = decoded[:, :, 0]
        flipped = np.asfortranarray(np.fliplr(decoded).astype(np.uint8))
        encoded = mask_utils.encode(flipped)
        encoded["counts"] = encoded["counts"].decode("ascii")
        output["segmentation"] = encoded
    elif isinstance(segmentation, list):
        polygons = []
        for polygon in segmentation:
            restored = list(polygon)
            for index in range(0, len(restored), 2):
                restored[index] = float(width - restored[index])
            polygons.append(restored)
        output["segmentation"] = polygons
    return output


def _bbox_iou_xywh(first: list[float], second: list[float]) -> float:
    ax1, ay1, aw, ah = first
    bx1, by1, bw, bh = second
    ax2, ay2 = ax1 + aw, ay1 + ah
    bx2, by2 = bx1 + bw, by1 + bh
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    union = aw * ah + bw * bh - intersection
    return intersection / union if union > 0 else 0.0


def _fuse_tta_predictions(
    view_files: list[tuple[Path, bool]],
    test_json: Path,
    destination: Path,
    iou_threshold: float,
    min_support: int,
) -> dict:
    import numpy as np
    from pycocotools import mask as mask_utils

    ground_truth = json.loads(test_json.read_text())
    image_sizes = {
        int(image["id"]): (int(image["width"]), int(image["height"]))
        for image in ground_truth["images"]
    }
    grouped: defaultdict = defaultdict(list)
    loaded_views = 0
    for view_id, (path, flipped) in enumerate(view_files):
        if not path.is_file():
            print(f"  [tta] missing prediction view: {path}")
            continue
        predictions = json.loads(path.read_text())
        loaded_views += 1
        for prediction in predictions:
            image_id = int(prediction["image_id"])
            width, _ = image_sizes[image_id]
            restored = (
                _flip_prediction_to_original(prediction, width)
                if flipped else dict(prediction)
            )
            restored["_view_id"] = view_id
            grouped[(image_id, int(restored["category_id"]))].append(restored)

    if loaded_views == 0:
        raise RuntimeError("No TTA prediction files were produced")

    fused = []
    supports = []
    for (image_id, category_id), detections in grouped.items():
        detections.sort(key=lambda item: float(item["score"]), reverse=True)
        clusters: list[list[dict]] = []
        for detection in detections:
            selected = None
            best_iou = iou_threshold
            for cluster in clusters:
                cluster_views = {item["_view_id"] for item in cluster}
                if detection["_view_id"] in cluster_views:
                    continue
                overlap = _bbox_iou_xywh(detection["bbox"], cluster[0]["bbox"])
                if overlap >= best_iou:
                    selected = cluster
                    best_iou = overlap
            if selected is None:
                clusters.append([detection])
            else:
                selected.append(detection)

        width, height = image_sizes[image_id]
        for cluster in clusters:
            support = len({item["_view_id"] for item in cluster})
            if support < min_support:
                continue
            masks = []
            weights = []
            for detection in cluster:
                segmentation = detection.get("segmentation")
                if not isinstance(segmentation, dict):
                    continue
                decoded = mask_utils.decode(segmentation)
                if decoded.ndim == 3:
                    decoded = decoded[:, :, 0]
                if decoded.shape != (height, width):
                    continue
                masks.append(decoded.astype(np.float32))
                weights.append(max(float(detection["score"]), 1e-6))
            if not masks:
                continue
            weight_array = np.asarray(weights, dtype=np.float32)
            probability = np.average(
                np.stack(masks, axis=0), axis=0, weights=weight_array
            )
            binary = np.asfortranarray((probability >= 0.5).astype(np.uint8))
            if not binary.any():
                best_segmentation = cluster[0]["segmentation"]
                binary = np.asfortranarray(
                    mask_utils.decode(best_segmentation).astype(np.uint8)
                )
            encoded = mask_utils.encode(binary)
            encoded["counts"] = encoded["counts"].decode("ascii")
            bbox = mask_utils.toBbox(encoded).astype(float).tolist()
            area = float(mask_utils.area(encoded))
            score = float(np.mean([float(item["score"]) for item in cluster]))
            fused.append({
                "image_id": image_id,
                "category_id": category_id,
                "bbox": bbox,
                "segmentation": encoded,
                "area": area,
                "score": score,
            })
            supports.append(support)

    destination.write_text(json.dumps(fused))
    support_counts = Counter(supports)
    print(
        f"  [tta] {loaded_views} views -> {len(fused)} fused instances; "
        f"support histogram={dict(sorted(support_counts.items()))}"
    )
    return {
        "views": loaded_views,
        "detections": len(fused),
        "support_histogram": {
            str(key): int(value) for key, value in sorted(support_counts.items())
        },
    }


def _coco_evaluate(ground_truth: Path, predictions: Path, names: list[str]) -> dict:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    detections = json.loads(predictions.read_text())
    if not detections:
        return {
            "segm_mAP50": 0.0,
            "segm_mAP50_95": 0.0,
            "per_class_mAP50": {name: 0.0 for name in names},
        }
    coco_gt = COCO(str(ground_truth))
    coco_dt = coco_gt.loadRes(str(predictions))
    evaluator = COCOeval(coco_gt, coco_dt, "segm")
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    per_class = {}
    for category_id, name in enumerate(names, start=1):
        class_evaluator = COCOeval(coco_gt, coco_dt, "segm")
        class_evaluator.params.catIds = [category_id]
        class_evaluator.evaluate()
        class_evaluator.accumulate()
        class_evaluator.summarize()
        per_class[name] = float(class_evaluator.stats[1])
    return {
        "segm_mAP50": float(evaluator.stats[1]),
        "segm_mAP50_95": float(evaluator.stats[0]),
        "per_class_mAP50": per_class,
    }


# %% 5. End-to-end Session 21 run
def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(
        description="SESSION 21: Mask2Former Swin-L single run with mask-aware TTA"
    )
    parser.add_argument("--data-root", default="auto")
    parser.add_argument("--work-dir", default=None)
    parser.add_argument("--device", default="auto", help="auto, 0, or 0,1")
    parser.add_argument("--max-iters", type=int, default=10000,
                        help="Training iterations; 10000 is the conservative "
                             "Kaggle default. Benchmark with --quick first.")
    parser.add_argument("--image-size", type=int, default=640)
    parser.add_argument("--accumulation", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--tta-scales", default="512,640,800")
    parser.add_argument("--tta-iou", type=float, default=0.55)
    parser.add_argument("--tta-min-support", type=int, default=1)
    parser.add_argument("--no-tta", action="store_true")
    parser.add_argument("--skip-train", action="store_true")
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--quick", action="store_true")
    args, _ = parser.parse_known_args(argv)

    if args.max_iters < 1:
        raise ValueError("--max-iters must be positive")
    if args.image_size < 256 or args.image_size % 32:
        raise ValueError("--image-size must be >=256 and divisible by 32")
    if args.accumulation < 1:
        raise ValueError("--accumulation must be positive")

    multiprocessing.freeze_support()
    started = time.time()
    print("=" * 72)
    print("SESSION 21 — Mask2Former Swin-L + mask-aware scale/flip TTA")
    print("=" * 72)
    print(f"  PYTORCH_ALLOC_CONF={os.environ['PYTORCH_ALLOC_CONF']}")

    if _ensure_supported_runtime():
        return 0
    _ensure_dependencies()
    gpu_ids = _detect_gpu_ids(args.device)
    print(f"  [gpu] visible training devices: {gpu_ids}")

    root = _find_dataset_root(args.data_root)
    names = _load_class_names(root / "data.yaml")
    train_items = _gather_split(root, "train")
    valid_items = _gather_split(root, "valid")
    # test split is optional — not all dataset versions include a test/ folder.
    try:
        test_items = _gather_split(root, "test")
    except (FileNotFoundError, ValueError):
        test_items = []

    if args.quick:
        train_items = _balanced_subset(train_items, 64, args.seed)
        valid_items = _balanced_subset(valid_items, 32, args.seed + 1)
        test_items = _balanced_subset(test_items, 12, args.seed + 2)
        args.max_iters = min(args.max_iters, 20)
        args.tta_scales = str(args.image_size)
        print(
            "  [quick] 64 train / 32 valid / 12 test, 20 iterations, "
            "one scale with flip"
        )

    default_work = Path("/kaggle/working") / (
        f"{SESSION_NAME}_quick" if args.quick else SESSION_NAME
    )
    work = Path(args.work_dir).expanduser().resolve() if args.work_dir else default_work
    work.mkdir(parents=True, exist_ok=True)
    coco_dir = work / "coco"
    config_dir = work / "configs"
    train_work = work / "train"
    prediction_dir = work / "predictions"
    for directory in (coco_dir, config_dir, train_work, prediction_dir):
        directory.mkdir(parents=True, exist_ok=True)

    print(f"  [data] classes: {names}")
    print(
        f"  [data] train={len(train_items)}, valid={len(valid_items)}, "
        f"locked test={len(test_items)}"
    )

    train_json = coco_dir / "train.json"
    valid_json = coco_dir / "valid.json"
    test_json = coco_dir / "test.json"
    conversion = {
        "train": _yolo_to_coco(train_items, names, train_json),
        "valid": _yolo_to_coco(valid_items, names, valid_json),
        "test": _yolo_to_coco(test_items, names, test_json),
    }

    pretrained = _download_and_prepare_coco_checkpoint(work)
    train_config = config_dir / "mask2former_swin_l_skin.py"
    _write_training_config(
        train_config,
        train_json,
        valid_json,
        train_work,
        pretrained,
        names,
        len(gpu_ids),
        args.max_iters,
        args.image_size,
        args.accumulation,
        args.seed,
    )

    # Config parsing catches inheritance-independent syntax mistakes before the
    # expensive launcher begins. Full base resolution is checked by train.py.
    compile(train_config.read_text(), str(train_config), "exec")
    best = _best_checkpoint(train_work)
    if args.skip_train:
        if best is None:
            raise FileNotFoundError(
                f"--skip-train was set but no checkpoint exists in {train_work}"
            )
        print(f"  [train] using existing checkpoint: {best}")
    else:
        print(
            f"  [train] starting {args.max_iters} iterations; batch/GPU=1, "
            f"gradient accumulation={args.accumulation}"
        )
        _run_training(train_config, train_work, gpu_ids, args.resume)
        best = _best_checkpoint(train_work)
        if best is None:
            raise RuntimeError("Training finished without producing a checkpoint")
        print(f"  [train] selected checkpoint: {best}")

    scales = sorted({
        int(value.strip()) for value in args.tta_scales.split(",") if value.strip()
    } | {args.image_size})
    if not scales:
        scales = [args.image_size]
    if any(scale < 256 or scale % 32 for scale in scales):
        raise ValueError("Every --tta-scales value must be >=256 and divisible by 32")
    views = [(args.image_size, False)] if args.no_tta else [
        (scale, flip) for scale in scales for flip in (False, True)
    ]

    print(f"  [test] locked-test prediction views: {views}")
    view_outputs: list[tuple[Path, bool]] = []
    normal_prediction = None
    for scale, flip in views:
        tag = f"scale{scale}_{'hflip' if flip else 'plain'}"
        config_path = config_dir / f"test_{tag}.py"
        prefix = prediction_dir / tag
        _write_test_config(
            train_config, config_path, test_json, prefix, scale, flip
        )
        expected = Path(f"{prefix}.segm.json")
        print(f"  [test] running {tag} ...")
        _run_prediction_pass(config_path, best, prediction_dir / tag, gpu_ids[0])
        if not expected.is_file():
            raise FileNotFoundError(
                f"MMDetection did not produce the expected prediction file: {expected}"
            )
        view_outputs.append((expected, flip))
        if scale == args.image_size and not flip:
            normal_prediction = expected

    if normal_prediction is None:
        normal_prediction = view_outputs[0][0]
    print("\n  [eval] normal single-view result")
    normal_metrics = _coco_evaluate(test_json, normal_prediction, names)

    tta_prediction = prediction_dir / "tta_mask_fused.segm.json"
    fusion_report = _fuse_tta_predictions(
        view_outputs,
        test_json,
        tta_prediction,
        args.tta_iou,
        args.tta_min_support,
    )
    print("\n  [eval] fused TTA result")
    tta_metrics = _coco_evaluate(test_json, tta_prediction, names)

    elapsed_hours = (time.time() - started) / 3600.0
    results = {
        "session": "SESSION_21",
        "model": "Mask2Former Swin-L",
        "method": "single training run plus mask-aware scale/horizontal-flip TTA",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_root": str(root),
        "class_names": names,
        "conversion": conversion,
        "training": {
            "checkpoint": str(best),
            "config": str(train_config),
            "max_iters": args.max_iters,
            "image_size": args.image_size,
            "gpus": gpu_ids,
            "batch_per_gpu": 1,
            "gradient_accumulation": args.accumulation,
            "quick": args.quick,
        },
        "tta": {
            "views": views,
            "iou_threshold": args.tta_iou,
            "minimum_support": args.tta_min_support,
            "fusion": fusion_report,
        },
        "normal_test": normal_metrics,
        "tta_test": tta_metrics,
        "elapsed_hours": elapsed_hours,
        "limitations": [
            "The expected mAP range in model_strategy_analysis.md is a planning "
            "estimate, not an empirical guarantee.",
            "MMDetection 3.3 plus MMCV 2.2 uses a narrowly scoped version-guard "
            "compatibility adjustment in the isolated Python-3.11 runtime.",
        ],
    }
    results_path = work / "SESSION_21_results.json"
    results_path.write_text(json.dumps(results, indent=2, default=str))

    archive = Path("/kaggle/working/SESSION_21_results.zip")
    if not Path("/kaggle/working").is_dir():
        archive = work / "SESSION_21_results.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as output_zip:
        for path in [
            results_path,
            train_config,
            normal_prediction,
            tta_prediction,
        ]:
            output_zip.write(path, arcname=path.name)

    print("\n" + "=" * 72)
    print("SESSION 21 COMPLETE")
    print("=" * 72)
    print(
        f"Normal test segm mAP50={normal_metrics['segm_mAP50']:.4f}, "
        f"mAP50-95={normal_metrics['segm_mAP50_95']:.4f}"
    )
    print(
        f"TTA test    segm mAP50={tta_metrics['segm_mAP50']:.4f}, "
        f"mAP50-95={tta_metrics['segm_mAP50_95']:.4f}"
    )
    print("Per-class TTA mAP50:")
    for name, value in tta_metrics["per_class_mAP50"].items():
        print(f"  {name}: {value:.4f}")
    print(f"Results JSON: {results_path}")
    print(f"Results ZIP : {archive}")
    print(f"Checkpoint  : {best}  (download separately from Kaggle outputs)")
    return 0


if __name__ == "__main__":
    multiprocessing.freeze_support()
    exit_code = main()
    try:
        sys.exit(exit_code)
    except SystemExit as error:
        # Notebook runners can treat sys.exit(0) as a cell error. Suppress only 0.
        if error.code:
            raise


SESSION 21 — Mask2Former Swin-L + mask-aware scale/flip TTA
  PYTORCH_ALLOC_CONF=expandable_segments:True
  [runtime] wrote notebook source to /kaggle/working/SESSION_21_runtime.py
  [runtime] creating seeded isolated Python 3.11 environment ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 79.2 MB/s eta 0:00:00


 Downloaded cpython-3.11.17-linux-x86_64-gnu (download)
Using CPython 3.11.17
Creating virtual environment with seed packages at: session21_openmmlab_py311
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
 + packaging==26.3
 + pip==26.2.1
 + setuptools==84.0.0
 + wheel==0.48.0
Activate with: source session21_openmmlab_py311/bin/activate


  [runtime] re-launching Session 21 with Python 3.11 ...
SESSION 21 — Mask2Former Swin-L + mask-aware scale/flip TTA
  PYTORCH_ALLOC_CONF=expandable_segments:True
  [deps] current environment probe failed:
Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'torch'
  [deps] installing isolated Python-3.11 OpenMMLab packages ...
  [install] --upgrade pip setuptools==75.1.0 wheel
  [install] --upgrade --force-reinstall --no-cache-dir torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
  [install] --upgrade numpy<2 addict termcolor packaging opencv-python-headless==4.10.0.84 matplotlib pyyaml Pillow pycocotools shapely six terminaltables tqdm scipy rich yapf
  [install] --upgrade --no-deps mmengine==0.10.5
  [install] --upgrade --no-cache-dir --only-binary=mmcv --no-deps mmcv==2.2.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4.0/index.html
  [install] --upgrade --no-deps mmdet==3.3.0
 